# 5. Measuring it

The first four notebooks build things. This one is about finding out
whether they work, which turned out to be a different skill.

Three dry runs over the same 50 leads, no messages sent. Every number
here is real. The interesting part is that the most expensive problem the
runs found was one the numbers never showed.


## Counting SMS segments

The first thing to measure was cost, and the first thing I got wrong was
how SMS billing works.

Twilio charges per *segment*, not per message. 160 characters per segment,
or 153 when a message spans several, because each part carries a header
saying which piece it is.


In [ ]:
def segments_naive(text):
    return 1 if len(text) <= 160 else -(-len(text) // 153)

for n in [150, 160, 161, 249, 306, 307]:
    print(f'{n:>4} chars -> {segments_naive("a" * n)} segments')


Looks right, and it is - for plain ASCII. Here is what it misses.

SMS uses a 7-bit alphabet called GSM-7 with about 128 characters. One
character outside it and the whole message switches to UCS-2, where each
character takes 16 bits instead of 7. The segment size drops from 160 to
70.

A curly apostrophe is outside GSM-7. A straight one is not.


In [ ]:
GSM = set(
    "@£$¥èéùìòÇØøÅåΔ_ΦΓΛΩΠΨΣΘΞÆæßÉ !\"#¤%&'()*+,-./0123456789:;<=>?"
    "¡ABCDEFGHIJKLMNOPQRSTUVWXYZÄÖÑÜ§¿abcdefghijklmnopqrstuvwxyzäöñüà"
    "\n\r\f"
) | set("^{}\\[~]|€")

def segments(text):
    if not text:
        return 0
    single, multi = (160, 153) if all(ch in GSM for ch in text) else (70, 67)
    return 1 if len(text) <= single else -(-len(text) // multi)

straight = "Don't miss a call. VoiceCaptures answers when you can't. " * 2
curly = straight.replace("'", "\u2019")

print(f'{len(straight)} chars, straight apostrophes -> {segments(straight)} segment(s)')
print(f'{len(curly)} chars, curly apostrophes    -> {segments(curly)} segment(s)')
print()
print('naive count would say:', segments_naive(curly), 'for both')


Same length, same words, one is billed at three times the other. And a
character count can't see it.

This isn't hypothetical - it happened. Five of 50 messages in the second
run were under 160 characters and still billing as three segments,
because the model wrote `don't` with a typographic apostrophe.

## Run one: the baseline

All five cold-pipeline agents on gpt-4o, drafting prompt saying "under
320 characters (about 2 SMS segments)".


In [ ]:
v1 = {'openai_per_prospect': 0.0062, 'p95_ms': 26050, 'failures': 1,
      'segments': {1: 0, 2: 50, 3: 0}, 'median_chars': 249,
      'picker': {'professional': 39, 'executive': 9, 'witty': 2}}

TWILIO_PER_SEGMENT = 0.0083

def summarize(run, label):
    n = sum(run['segments'].values())
    segs = sum(k * v for k, v in run['segments'].items())
    twilio = segs / n * TWILIO_PER_SEGMENT
    combined = run['openai_per_prospect'] + twilio
    print(f'{label}')
    print(f"  openai   ${run['openai_per_prospect']:.4f}/prospect")
    print(f'  twilio   ${twilio:.4f}/prospect  ({segs / n:.2f} segments avg)')
    print(f'  combined ${combined:.4f}  ->  ${combined * 484:.2f} for 484 leads')
    print(f"  p95 {run['p95_ms'] / 1000:.1f}s, {run['failures']} failure(s)")
    return combined

c1 = summarize(v1, 'v1  gpt-4o, 320 char ceiling')


Two problems visible immediately.

Every message is two segments. The prompt literally said "about 2 SMS
segments" - someone had accepted paying double and written it into the
instructions.

And p95 of 26 seconds with a failure. That one isn't slowness:


In [ ]:
print('RateLimitError: Rate limit reached for gpt-4o ... on tokens per min')
print('(TPM): Limit 30000, Used 29828, Requested 490.')
print()
print('5 concurrent prospects x 5 gpt-4o calls x ~1600 tokens =',
      5 * 5 * 1600, 'tokens in flight')


Self-inflicted. The pipeline was being throttled by its own model choice.

## Run two: cheap model, tight ceiling

Moved all five agents to gpt-4o-mini and dropped the length ceiling to
120 characters for the message body - the 36-character compliance footer
is appended afterwards, and the total has to clear 160.


In [ ]:
v2 = {'openai_per_prospect': 0.0004, 'p95_ms': 5466, 'failures': 0,
      'segments': {1: 38, 2: 7, 3: 5}, 'median_chars': 150,
      'picker': {'professional': 0, 'executive': 9, 'witty': 41}}

c2 = summarize(v2, 'v2  gpt-4o-mini, 120 char ceiling')
print(f'\nopenai cost fell {v1["openai_per_prospect"] / v2["openai_per_prospect"]:.0f}x')


16x cheaper on OpenAI, rate limiting gone. But look at the segments: 5
messages went to *three*, worse than the baseline. Those are the curly
apostrophes.

And the combined cost barely moved, because Twilio now dominates. Once
the model is cheap, the messaging bill is the whole cost.

## Run three: fix it in code, not the prompt

The drafting prompt already said "straight apostrophes only". The model
did it anyway. Same class of problem as the compliance footer in notebook
4: anything with a cost attached can't depend on an instruction being
followed.


In [ ]:
SUBS = {'\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"',
        '\u2013': '-', '\u2014': '-', '\u2026': '...', '\u00a0': ' '}

def sanitize(text):
    for bad, good in SUBS.items():
        text = text.replace(bad, good)
    text = ' '.join(text.split())            # collapse stray newlines
    if len(text) >= 2 and text[0] == text[-1] and text[0] in '"\'':
        text = text[1:-1].strip()            # models sometimes quote the whole thing
    return text

# Real drafts from the v2 run, with the compliance footer appended as it
# is at send time.
FOOTER = ' Reply YES for a demo, NO to opt-out.'
real = [
    'With a 4.8 rating from 150 reviews, don\u2019t miss calls that could become jobs!' + FOOTER,
    'Your perfect 5.0 rating with just 3 reviews suggests you\u2019re missing job opportunities.' + FOOTER,
    '"Never miss a weekend plumbing job! VoiceCaptures can handle calls.\n  " + FOOTER',
]
saved = 0
for ex in real[:2]:
    b, a = segments(ex), segments(sanitize(ex))
    saved += b - a
    print(f'{len(ex):>3} chars   {b} seg -> {a} seg   {sanitize(ex)[:55]}...')
print(f'\n{saved} segments saved on 2 messages, at ${TWILIO_PER_SEGMENT} each')


In [ ]:
v3 = {'openai_per_prospect': 0.0004, 'p95_ms': 5254, 'failures': 0,
      'segments': {1: 47, 2: 3, 3: 0}, 'median_chars': 141,
      'picker': {'professional': 1, 'executive': 12, 'witty': 37}}

c3 = summarize(v3, 'v3  + sanitizer + forbidden angles')
print()
for label, c in [('v1', c1), ('v2', c2), ('v3', c3)]:
    print(f'{label}  ${c * 484:>6.2f} for 484 leads')


Zero three-segment messages. 94% at one. Combined cost down 60% from the
baseline.

## The finding the numbers missed

After run two every metric looked healthy - cheap, fast, no failures. So
I read the actual messages, which I had not done at all up to that point.

> XL Services: "Your rating is 1.0 with just 1 review - let
> VoiceCaptures help turn calls into jobs!"

> TNR Gas & Plumbing: "Your 2.7 rating with 7 reviews shows missed job
> opportunities."

Both true. Both grounded in real data, exactly as the hook agent was
designed to do. Both a cold text from a stranger telling a business owner
their reputation is bad.

Nothing in the system was broken. The hook agent was told to find the
sharpest angle in the data and it did. Nobody had told it that some true
facts are unusable.

Worse: tightening the length ceiling made it more brutal, because the
softening qualifier is the first thing cut when you have 120 characters.
The optimization made the failure sharper.


In [ ]:
before = {
    'XL Services': 'Your rating is 1.0 with just 1 review - let VoiceCaptures help!',
    'TNR Gas': 'Your 2.7 rating with 7 reviews shows missed job opportunities.',
}
after = {
    'XL Services': 'Missing calls could be costing XL Services jobs. VoiceCaptures helps!',
    'TNR Gas': 'Missed calls can cost you plumbing jobs. VoiceCaptures ensures 24/7 coverage.',
}
for k in before:
    print(f'{k}\n  before: {before[k]}\n  after:  {after[k]}\n')


The fix was a forbidden-angles list in the hook agent - no rating below
4.0, no low review count framed as a deficiency on its own, nothing
implying the business is failing. With a test written into the prompt:
read the angle back as the owner, on a Tuesday, from a number you don't
recognise. If any part of it stings, pick a different angle.

### It held 49 times out of 50

One message still referenced a 3.9 rating - and called it "impressive",which is the model straining to obey the tone rule while breaking the
content rule.

Same lesson as the apostrophes, one level up. A prompt rule gets ~98%.
The remaining 2% needs code: strip sub-4.0 ratings from the data before
the hook agent sees them, rather than asking it not to look. Not done
yet, and it should be before any real send.

## The measurement that measured nothing

One more thing the runs surfaced, and it undermines an experiment I
thought I had running.


In [ ]:
for label, run in [('v1', v1), ('v2', v2), ('v3', v3)]:
    n = sum(run['picker'].values())
    parts = '  '.join(f'{k[:4]} {v / n:>4.0%}' for k, v in run['picker'].items())
    print(f'{label}  {parts}')


Professional won 78% of picks, then 0%, then 2%. Not a shift - an
inversion, with the previous winner never winning again.

I changed the model and the length ceiling in the same run, so the cause
is unattributable. That is my error; the runs should have been split.

But reading the picker's stated reasons is what actually worries me:

> "compliments my business while maintaining clarity"
> "resonates with my business needs"

First person, as the prospect. It is roleplaying the recipient rather
than evaluating copy against a standard. That would explain preferences
flipping wholesale under a change to the drafting setup - and it means
persona win rate is a much weaker signal than I assumed. The experiment
that was supposed to tell me which style works may not be measuring style
at all.

## What I'd take from this

**Measure before optimizing, and change one thing per run.** I know this.
I changed two and lost the ability to explain the most interesting result
in the data.

**Cheap models move the bottleneck rather than removing it.** After the
16x OpenAI reduction, Twilio was 96% of the cost. The optimization that
mattered second was the one I nearly skipped.

**A metric that looks healthy is not a system that works.** Every number
after run two was good. The messages were insulting people.

**Read the outputs.** Not a sample of the summary statistics - the actual
text that a human is going to receive. Ten minutes reading 50 messages
found the worst problem in the project, and no dashboard would have.

## What this became

| here | in the repo |
|---|---|
| `segments` | `segments()` in `app/measure_cold.py` |
| `sanitize` | `sanitize_for_sms()` in `app/tools/twilio_sms.py`, called inside `send_sms` |
| forbidden angles | the NEVER-use list in `app/agents/hook_agent.py` |
| the run summaries | `python -m app.measure_cold --limit 50 --out runs/x.csv` |

Token counting in the real script works by wrapping `Runner.run` for the
duration - the pipeline returns a dict and discards the `RunResult`
objects that carry usage, so there is nothing to read afterwards. The
wrapper lives in the measurement script rather than in `cold_outreach.py`,
because a measurement tool should not require changing the thing it
measures.
